# Part 3: Recommendation, Explainability, and Cold Start Strategy
## Netflix Prize Dataset - Personalized Discovery & Explanations

This notebook demonstrates:
1. Loading the saved SVD and KNN models and movie metadata.
2. Generating Top-K recommendations for arbitrary users.
3. Designing and implementing an **Explainability Framework** to explain recommendations using item-to-item similarities.
4. Formulating and executing a **Cold Start Strategy** for new users, new movies, and sparse user histories.

In [1]:
import os
import pickle
import pandas as pd
import numpy as np

print("Libraries loaded successfully.")

Libraries loaded successfully.


### 1. Load Serialized Models and Data

In [2]:
models_dir = os.path.join("..", "backend", "models")
ratings_path = os.path.join("..", "data", "processed", "ratings_subset.csv")

# Load SVD Model
with open(os.path.join(models_dir, "svd_model.pkl"), "rb") as f:
    svd_model = pickle.load(f)

# Load KNN Model
with open(os.path.join(models_dir, "knn_model.pkl"), "rb") as f:
    knn_model = pickle.load(f)

# Load Movie Meta
with open(os.path.join(models_dir, "movies_meta.pkl"), "rb") as f:
    df_movies = pickle.load(f)

# Load Ratings subset to construct user profiles
df_ratings = pd.read_csv(ratings_path)

print("Models and data subsets successfully loaded.")

Models and data subsets successfully loaded.


### 2. Generate Top-K Recommendations

To recommend movies to an existing user:
- Find all movies the user has *not* yet rated in our dataset.
- Use the SVD model to predict ratings for these unseen movies.
- Sort predictions in descending order and return the top K movies.

In [3]:
def get_user_recommendations(user_id, k=10):
    # Get all movie IDs present in dataset
    all_movie_ids = df_movies["movie_id"].unique()
    
    # Get movies the user has already rated
    user_history = df_ratings[df_ratings["user_id"] == user_id]
    rated_movie_ids = user_history["movie_id"].values
    
    # Identify unrated movies
    unrated_movie_ids = [m_id for m_id in all_movie_ids if m_id not in rated_movie_ids]
    
    # Predict ratings for all unrated movies using SVD
    predictions = []
    for m_id in unrated_movie_ids:
        pred = svd_model.predict(user_id, m_id)
        predictions.append((m_id, pred.est))
        
    # Sort predictions by estimated rating descending
    predictions.sort(key=lambda x: x[1], reverse=True)
    top_predictions = predictions[:k]
    
    # Format output as a DataFrame with movie metadata
    rec_df = pd.DataFrame(top_predictions, columns=["movie_id", "predicted_rating"])
    rec_df = rec_df.merge(df_movies, on="movie_id")
    return rec_df[["movie_id", "title", "year", "predicted_rating"]]

# Sample recommendation for user 134001
sample_user = 134001
recs = get_user_recommendations(sample_user, k=5)
print(f"Top 5 Recommendations for User {sample_user}:")
print(recs.to_string(index=False))

Top 5 Recommendations for User 134001:
 movie_id                                         title  year  predicted_rating
     3456                                Lost: Season 1  2004          4.644315
     2452 Lord of the Rings: The Fellowship of the Ring  2001          4.357767
     2102                        The Simpsons: Season 6  1994          4.305360
     3962                     Finding Nemo (Widescreen)  2003          4.277521
     4306                               The Sixth Sense  1999          4.241847


### 3. Explainable Recommendations Framework

To provide an explanation for each recommendation, we use our item-based similarity matrix from the KNN model:
- For a recommended movie $M$, we identify its top neighbors (similar movies) using the similarity weights in the KNN model.
- We cross-reference these neighbors with the user's historical ratings.
- We identify the movie $M_{watched}$ in the user's history that had the highest similarity to $M$ and a high rating (e.g., rating $\ge 4$).
- We formulate a text explanation: *"Recommended because you rated '{M_watched}' a {rating} (Similarity: {score}%)"*.

In [4]:
def explain_recommendation(user_id, rec_movie_id, top_n_explanations=1):
    # 1. Get user history with high ratings
    user_history = df_ratings[(df_ratings["user_id"] == user_id) & (df_ratings["rating"] >= 4.0)]
    if user_history.empty:
        return "Recommended because it is highly rated by similar users."
        
    # 2. Get internal Surprise item ID for the recommended movie
    try:
        rec_inner_id = knn_model.trainset.to_inner_iid(rec_movie_id)
    except ValueError:
        # Movie not in training set (cold start)
        return "Recommended because of its overall popularity."
        
    # 3. Find similarities between the recommended movie and user's high-rated movies
    similarities = []
    for _, row in user_history.iterrows():
        hist_movie_id = int(row["movie_id"])
        rating = row["rating"]
        try:
            hist_inner_id = knn_model.trainset.to_inner_iid(hist_movie_id)
            # Retrieve similarity from KNN matrix
            sim = knn_model.sim[rec_inner_id, hist_inner_id]
            similarities.append((hist_movie_id, sim, rating))
        except ValueError:
            continue
            
    if not similarities:
        return "Recommended because it matches your overall movie profile."
        
    # Sort similarities by score descending
    similarities.sort(key=lambda x: x[1], reverse=True)
    similar_movie_id, sim_score, rating = similarities[0]
    
    # Retrieve names
    watched_title = df_movies[df_movies["movie_id"] == similar_movie_id]["title"].values[0]
    rec_title = df_movies[df_movies["movie_id"] == rec_movie_id]["title"].values[0]
    
    explanation = (
        f"We recommend '{rec_title}' because you gave '{watched_title}' "
        f"a rating of {rating} (Similarity: {sim_score*100:.1f}%)."
    )
    return explanation

# Test explanation on sample user recommendations
print(f"Explanations for recommendations for User {sample_user}:")
for _, row in recs.iterrows():
    expl = explain_recommendation(sample_user, int(row["movie_id"]))
    print(f"- {expl}")

Explanations for recommendations for User 134001:
- Recommended because it is highly rated by similar users.


- Recommended because it is highly rated by similar users.
- Recommended because it is highly rated by similar users.
- Recommended because it is highly rated by similar users.


- Recommended because it is highly rated by similar users.


### 4. Cold Start Strategies

#### 4.1. New Users
For new users with no historical ratings, we cannot perform SVD or user/item filtering. Our fallback is a **Popularity-based Recommender** which returns the movies with the highest average rating among movies that have received at least 1,000 ratings.

In [5]:
def get_cold_start_recommendations(k=10):
    # Group by movie, get counts and means
    movie_stats = df_ratings.groupby("movie_id")["rating"].agg(["count", "mean"])
    # Filter to movies with high number of ratings
    popular_movies = movie_stats[movie_stats["count"] >= 2000]
    # Sort by mean rating descending
    top_popular = popular_movies.sort_values(by="mean", ascending=False).head(k)
    
    rec_df = top_popular.reset_index().merge(df_movies, on="movie_id")
    return rec_df[["movie_id", "title", "year", "mean"]].rename(columns={"mean": "avg_rating"})

print("Popularity Fallback recommendations for Cold Start Users:")
print(get_cold_start_recommendations(5).to_string(index=False))

Popularity Fallback recommendations for Cold Start Users:


 movie_id                                         title  year  avg_rating
     2102                        The Simpsons: Season 6  1994    4.509460
     3444         Family Guy: Freakin' Sweet Collection  2004    4.434948
     2452 Lord of the Rings: The Fellowship of the Ring  2001    4.426156
     2172                        The Simpsons: Season 3  1991    4.385802
     1256                   The Best of Friends: Vol. 4  1994    4.371876


#### 4.2. New Movies
When a new movie $M_{new}$ enters the system (having no ratings), SVD and Collaborative Filtering cannot make predictions for it (the Collaborative Filtering Cold Start problem).

**Our Strategy:**
1. **Genre Mapping**: If external metadata is available (e.g. TMDB genre mapping), we assign the movie similarity scores based on overlap with existing popular movies in the same genre.
2. **Exploration Promotion**: We inject a small fraction of newly added movies into the active recommendations list for random users (often called an $\epsilon$-greedy exploration strategy) to collect initial ratings and overcome the cold start.